# 4B. Keras-only CBOW - Twitter
True CBOW in Keras: generate (context, target) pairs from a sliding window, train an
`Embedding -> GlobalAveragePooling1D -> Dense(vocab, softmax)` model, then extract
the Embedding weights to inspect similar words.

In [1]:
import numpy as np
import pandas as pd
import re
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
df = pd.read_csv("../../datasets/CBOW/twitter_dataset.csv")
texts = df["Text"].dropna().tolist()

sentences = []
for tweet in texts:
    tokens = re.findall(r"[a-z]+", str(tweet).lower())
    if len(tokens) >= 5:
        sentences.append(tokens)

print("tweets kept:", len(sentences))

tweets kept: 10000


In [3]:
# build vocabulary (manual, so we can skip padding)
vocab = {}
for sent in sentences:
    for word in sent:
        if word not in vocab:
            vocab[word] = len(vocab)
vocab_size = len(vocab)
print("vocab:", vocab_size)

vocab: 971


In [4]:
# generate (context, target) pairs with window=2
window = 2
contexts = []
targets = []
for sent in sentences:
    ids = [vocab[w] for w in sent]
    for i in range(window, len(ids) - window):
        context = ids[i-window:i] + ids[i+1:i+window+1]
        contexts.append(context)
        targets.append(ids[i])

contexts = np.array(contexts)
targets = np.array(targets)
print("pairs:", contexts.shape, targets.shape)

pairs: (299853, 4) (299853,)


In [5]:
model = keras.Sequential([
    layers.Input(shape=(2*window,)),
    layers.Embedding(vocab_size, 50, name="embed"),
    layers.GlobalAveragePooling1D(),
    layers.Dense(vocab_size, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embed (Embedding)               │ (None, 4, 50)          │        48,550 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 50)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 971)            │        49,521 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 98,071 (383.09 KB)

 Trainable params: 98,071 (383.09 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
history = model.fit(contexts, targets, epochs=3, batch_size=256)

Epoch 1/3
   1/1172 ━━━━━━━━━━━━━━━━━━━━ 6:06 313ms/step - loss: 6.8784
  36/1172 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 6.8784
  73/1172 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 6.8784
 110/1172 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 6.8785
 147/1172 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 6.8785
 182/1172 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 6.8785
 215/1172 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 6.8785
 248/1172 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 6.8785
 282/1172 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 6.8785
 316/1172 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 6.8785
 351/1172 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 6.8785
 384/1172 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 6.8785
 419/1172 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 6.8785
 453/1172 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 6.8785
 487/1172 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 6.8785
 521/1172 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 6.8785
 555/1172 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 6.8785


In [7]:
# extract learned embeddings and find similar words via cosine similarity
embeddings = model.get_layer("embed").get_weights()[0]
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
embeddings_norm = embeddings / (norms + 1e-9)

id_to_word = {i: w for w, i in vocab.items()}

def most_similar(word, k=5):
    if word not in vocab:
        print(f"{word!r} not in vocab")
        return
    i = vocab[word]
    sims = embeddings_norm @ embeddings_norm[i]
    top = sims.argsort()[::-1][1:k+1]
    print(f"\nsimilar to {word!r}:")
    for j in top:
        print(f"  {id_to_word[j]}: {sims[j]:.3f}")

for word in ["love", "happy", "day"]:
    most_similar(word)

'love' not in vocab

similar to 'happy':
  nation: 0.474
  these: 0.468
  about: 0.468
  nothing: 0.442
  morning: 0.440

similar to 'day':
  son: 0.390
  try: 0.383
  goal: 0.381
  just: 0.366
  rock: 0.360
